# Evaluación del Sistema RAG Legal - TFM
## Análisis de Rendimiento y Métricas de Precisión

**Autor:** Pablo Fernández Lucas
**Fecha:** Septiembre 2025  
**Universidad:** Universidad Complutense de Madrid

### Resumen Ejecutivo
Este notebook presenta la evaluación completa del sistema RAG (Retrieval-Augmented Generation) desarrollado para consultas legales. Se analiza su precisión, eficiencia y capacidad de recuperación de información jurídica relevante.

### Objetivos
- Evaluar la precisión del sistema RAG en la recuperación de leyes y artículos específicos
- Analizar el rendimiento con diferentes configuraciones de parámetros
- Generar métricas de calidad para validar la efectividad del sistema
- Identificar áreas de mejora y optimización


## 1. Configuración Inicial y Librerías


In [ ]:
# === LIBRERÍAS PRINCIPALES ===
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
from datetime import datetime
from typing import List
from collections import Counter

# === LIBRERÍAS ESPECÍFICAS DEL PROYECTO ===
from utils.models.law_item_response import LawItemWithLaw, LawItemSimplify
from utils.repository.rag_repository import RAGService, RAGConfig
from utils.repository.openai_repository import OpenAIRepository
from utils.repository.supabase_repository import SupabaseRepository

# === LIBRERÍAS PARA VISUALIZACIÓN ===
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.text import Text
from rich import box

# === CONFIGURACIÓN INICIAL ===
env_path = ".env"
load_dotenv(dotenv_path=env_path)

# Configuración de matplotlib para mejores gráficos
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✅ Configuración inicial completada")
print(f"📅 Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


## 2. Modelos y Funciones Auxiliares


In [ ]:
from pydantic import BaseModel, Field, ConfigDict


class LegalQuizResponse(BaseModel):
    """
    Modelo para respuestas de quiz legal generadas por el sistema RAG.
    Utilizado para evaluar la capacidad del sistema de generar preguntas
    estructuradas con referencias legales precisas.
    """
    question: str = Field(description="La pregunta formulada")
    answer1: str = Field(description="Primera opción de respuesta")
    answer2: str = Field(description="Segunda opción de respuesta")
    answer3: str = Field(description="Tercera opción de respuesta")
    solution: int = Field(description="Índice de la respuesta correcta (1, 2 o 3)", ge=1, le=3)
    cites: List[LawItemSimplify] = Field(
        description="Citas legales usadas completas y con identificador de ley",
        default_factory=list
    )
    tip: str = Field(description="Argumentación de por qué es correcta", default="")



print("✅ Modelos definidos correctamente")


## 3. Ejemplos de Uso del Sistema RAG

En esta sección se demuestra el funcionamiento básico del sistema RAG con ejemplos prácticos.


### 3.1 Configuración del Servicio RAG


In [ ]:
# Inicialización del servicio RAG con configuración estándar
rag_service = RAGService(
    config=RAGConfig(
        model_name="gpt-5",
        agent_timeout=120,
        search_limit=20,
    )
)


### 3.2 Ejemplo con Respuesta Estándar


In [ ]:
# Ejemplo de consulta sobre derecho a la educación
query_ejemplo = "Citame las leyes que hablen sobre el derecho a la educación en España, cita 3 artículos y explica brevemente su contenido"

print(f"🔍 Procesando consulta: {query_ejemplo}")
print("⏳ Por favor, espere...")

result, metadata = await rag_service.process_query(query_ejemplo)

print("\n" + "=" * 80)
print("📋 RESPUESTA GENERADA")
print("=" * 80)
print(result.response)

print("\n" + "=" * 80)
print("🧠 RAZONAMIENTO DEL SISTEMA")
print("=" * 80)
print(result.reasoning)

print("\n" + "=" * 80)
print("📚 CITAS UTILIZADAS")
print("=" * 80)
for i, citation in enumerate(result.citations, 1):
    print(f"{i}. {citation}")

print(f"\n✅ Consulta procesada exitosamente")
print(f"📊 Metadatos: {len(result.citations)} citas encontradas")


### 3.3 Ejemplo con Respuesta Personalizada (Quiz Legal)


In [ ]:
# Configuración para generar quiz legal
config_quiz = RAGConfig(
    model_name="gpt-5",
    agent_timeout=120,
    response_model=LegalQuizResponse
)

rag_service_quiz = RAGService(config=config_quiz)

print("🎯 Generando quiz legal sobre el Código Civil artículo 2...")

result_quiz, metadata_quiz = await rag_service_quiz.process_query(
    "Genera una pregunta sobre el código civil artículo 2"
)

print("\n" + "=" * 60)
print("🎲 QUIZ LEGAL GENERADO")
print("=" * 60)
print(f"❓ Pregunta: {result_quiz.question}")
print(f"\n📝 Opciones:")
print(f"   1) {result_quiz.answer1}")
print(f"   2) {result_quiz.answer2}")
print(f"   3) {result_quiz.answer3}")
print(f"\n✅ Respuesta correcta: Opción {result_quiz.solution}")
print(f"💡 Explicación: {result_quiz.tip}")

if result_quiz.cites:
    print(f"\n📚 Referencias legales utilizadas:")
    for i, cite in enumerate(result_quiz.cites, 1):
        print(f"   {i}. {cite}")




## 4. Generación del Dataset de Evaluación

Para evaluar el sistema RAG de manera sistemática, generamos un dataset de preguntas basadas en textos legales reales.


### 4.1 Obtención de Leyes desde la Base de Datos


In [ ]:
# Conexión a la base de datos y obtención de muestra de leyes
print("🔗 Conectando a la base de datos...")
supabase_client = SupabaseRepository()

# Obtener una muestra representativa de leyes
print("📚 Obteniendo muestra de leyes aleatorias...")
laws_sample = supabase_client.get_random_laws(limit=1000)

print(f"✅ Obtenidas {len(laws_sample)} leyes de la base de datos")
print(f"📊 Distribución por tipo de contenido:")

# Análisis básico de la muestra
law_titles = [law.law_title for law in laws_sample]
title_counts = Counter(law_titles)
top_5_laws = title_counts.most_common(5)

for law_title, count in top_5_laws:
    print(f"   • {law_title[:50]}...: {count} artículos")


### 4.1.1 Visualización de los datos obtenidos en un grafo

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

def create_clean_graph(laws_sample, similarity_threshold=0.6):
    """
    Crea un grafo de similitud limpio desde cero
    """
    # 1. Extraer datos
    vectors = []
    nodes = []
    law_ids = []
    law_names = []  # Para la leyenda

    for law in laws_sample:
        if hasattr(law, 'vector') and law.vector is not None:
            vectors.append(law.vector)
            nodes.append(law.law_item_id)
            law_ids.append(law.law_id)
            # Intentar obtener el nombre de la ley
            if hasattr(law, 'name'):
                law_names.append(law.name)
            elif hasattr(law, 'title'):
                law_names.append(law.title)
            else:
                law_names.append(f"Ley {law.law_id}")

    if len(vectors) < 2:
        print("Error: Se necesitan al menos 2 vectores")
        return None

    vectors = np.array(vectors)
    print(f"✓ Procesando {len(vectors)} nodos")

    # 2. Calcular matriz de similitud coseno
    similarity_matrix = cosine_similarity(vectors)
    print(f"✓ Matriz de similitud calculada")

    # 3. Crear grafo
    G = nx.Graph()

    # Agregar nodos con metadata
    for i, node in enumerate(nodes):
        G.add_node(node,
                  law_id=law_ids[i],
                  law_name=law_names[i])

    # Agregar aristas solo si superan el umbral
    edges_added = 0
    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            sim = similarity_matrix[i, j]
            if sim > similarity_threshold:
                G.add_edge(nodes[i], nodes[j], weight=sim)
                edges_added += 1

    print(f"✓ {edges_added} conexiones creadas (similitud > {similarity_threshold})")

    # 4. Limpiar nodos problemáticos
    original_nodes = G.number_of_nodes()

    # Quitar nodos aislados
    isolated = list(nx.isolates(G))
    G.remove_nodes_from(isolated)

    # Quitar nodos con solo 1 conexión (opcionales)
    low_degree = [n for n, d in G.degree() if d == 1]
    G.remove_nodes_from(low_degree[:len(low_degree)//2])  # Solo la mitad

    removed = original_nodes - G.number_of_nodes()
    if removed > 0:
        print(f"✓ {removed} nodos problemáticos eliminados")

    print(f"✓ Grafo final: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas")
    return G

def plot_graph_with_legend(G):
    """
    Dibuja el grafo con leyenda completa de leyes
    """
    if G is None or G.number_of_nodes() == 0:
        print("Error: Grafo vacío")
        return

    # Preparar colores únicos por ley
    law_data = {}
    for node in G.nodes():
        law_id = G.nodes[node]['law_id']
        law_name = G.nodes[node]['law_name']
        if law_id not in law_data:
            law_data[law_id] = law_name

    # Asignar colores
    unique_laws = list(law_data.keys())
    colors = plt.cm.Set3(np.linspace(0, 1, len(unique_laws)))
    color_map = {law_id: colors[i] for i, law_id in enumerate(unique_laws)}

    # Colores para cada nodo
    node_colors = [color_map[G.nodes[node]['law_id']] for node in G.nodes()]

    # Layout optimizado
    pos = nx.spring_layout(G,
                          k=2/np.sqrt(G.number_of_nodes()),
                          iterations=500,
                          weight='weight',
                          seed=42)

    # Crear figura más grande para la leyenda
    plt.figure(figsize=(16, 10))

    # Dibujar nodos
    nx.draw_networkx_nodes(G, pos,
                          node_color=node_colors,
                          node_size=300,
                          alpha=0.8)

    # Dibujar aristas con grosor proporcional al peso
    if G.edges():
        edges = G.edges(data=True)
        weights = [d['weight'] for u, v, d in edges]
        nx.draw_networkx_edges(G, pos,
                              width=[w * 4 for w in weights],
                              alpha=0.6,
                              edge_color='gray')

    # Título
    plt.title(f'Red de Similitud de Leyes\n{G.number_of_nodes()} artículos, {G.number_of_edges()} conexiones',
             fontsize=16, fontweight='bold', pad=20)

    # Leyenda detallada
    legend_elements = []
    for law_id in sorted(unique_laws)[:5]:
        law_name = law_data[law_id]
        # Truncar nombres muy largos
        display_name = law_name if len(law_name) <= 40 else law_name[:37] + "..."
        legend_elements.append(
            plt.Line2D([0], [0], marker='o', color='w',
                      markerfacecolor=color_map[law_id],
                      markersize=12,
                      label=f'{law_id}: {display_name}')
        )

    # Colocar leyenda fuera del gráfico
    plt.legend(handles=legend_elements,
              loc='center left',
              bbox_to_anchor=(1.05, 0.5),
              fontsize=10,
              title="Leyes en la Red",
              title_fontsize=12)

    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # Estadísticas
    print(f"\n📊 ESTADÍSTICAS DEL GRAFO:")
    print(f"   • Nodos (artículos): {G.number_of_nodes()}")
    print(f"   • Conexiones: {G.number_of_edges()}")
    print(f"   • Leyes diferentes: {len(unique_laws)}")
    print(f"   • Densidad: {nx.density(G):.3f}")

    # Mostrar qué leyes están más conectadas
    law_connections = {}
    for law_id in unique_laws:
        nodes_of_law = [n for n in G.nodes() if G.nodes[n]['law_id'] == law_id]
        connections = sum(G.degree(n) for n in nodes_of_law)
        law_connections[law_id] = connections

    print(f"\n🔗 CONECTIVIDAD POR LEY:")
    for law_id, connections in sorted(law_connections.items(), key=lambda x: x[1], reverse=True):
        print(f"   • Ley {law_id}: {connections} conexiones ({law_data[law_id][:30]}...)")

# Función principal - todo en uno
def analyze_laws_similarity(laws_sample, similarity_threshold=0.6):
    """
    Analiza y visualiza la similitud entre las 5 leyes más numerosas

    Args:
        laws_sample: Lista de objetos con .vector, .law_item_id, .law_id
        similarity_threshold: Umbral de similitud (0.0 a 1.0)
    """
    print(f"🔍 ANALIZANDO LAS 5 LEYES MÁS NUMEROSAS (umbral: {similarity_threshold})")
    print("=" * 65)

    # Crear grafo limpio con solo top 5 leyes
    G = create_clean_graph(laws_sample, similarity_threshold)

    if G is not None:
        # Visualizar con leyenda
        plot_graph_with_legend(G)
        return G
    else:
        print("❌ No se pudo crear el grafo")
        return None

# === EJEMPLO DE USO ===
G = analyze_laws_similarity(laws_sample, similarity_threshold=0.6)

# Si quieres probar diferentes umbrales:
# for threshold in [0.4, 0.6, 0.8]:
#     print(f"\n{'='*15} UMBRAL {threshold} - TOP 5 LEYES {'='*15}")
#     G = analyze_laws_similarity(laws_sample, threshold)

In [ ]:
G.number_of_nodes(), G.number_of_edges()

### 4.2 Generación de Preguntas con OpenAI


In [ ]:
# Inicialización del cliente OpenAI para generar preguntas
openai_client = OpenAIRepository()

print("🤖 Iniciando generación de preguntas con OpenAI...")
print("⏳ Este proceso puede tomar varios minutos...")

results_generation = []
errors_count = 0

for i, law in enumerate(laws_sample[:20], 1):
    try:
        # Sistema prompt optimizado para generar preguntas realistas
        system_prompt = (
            "Eres un estudiante de derecho que formula consultas "
            "para poner a prueba un sistema RAG. Genera una sola pregunta "
            "o caso práctico breve basado en el texto legal, "
            "Hazlo en menos de dos frases, de forma natural y no obvia. "
            "La pregunta debe ser específica del contenido proporcionado."
            "Puedes nombrar la ley explicitamente y el numero de artículo."
        )

        response = openai_client.generate_text(
            system=system_prompt,
            prompt=f"Texto legal: {law.content}"
        )

        results_generation.append({
            "law": law,
            "response": response
        })

        # Mostrar progreso cada 100 elementos
        if i % 100 == 0:
            print(f"   ✅ Procesadas {i}/{len(laws_sample)} leyes...")

    except Exception as e:
        errors_count += 1
        print(f"   ⚠️ Error en ley {i}: {str(e)}")
        continue

print(f"\n🎉 Generación completada!")
print(f"   • Preguntas generadas: {len(results_generation)}")
print(f"   • Errores encontrados: {errors_count}")
print(f"   • Tasa de éxito: {(len(results_generation) / len(laws_sample)) * 100:.1f}%")


### 4.3 Creación y Guardado del Dataset


In [ ]:
# Creación del DataFrame con los datos generados
print("📊 Creando dataset estructurado...")

df_evaluation = pd.DataFrame({
    "query": [r["response"] for r in results_generation],
    "law_id": [r["law"].law_id for r in results_generation],
    "law_title": [r["law"].law_title for r in results_generation],
    "law_item_id": [r["law"].law_item_id for r in results_generation],
    "law_item_title": [r["law"].law_item_title for r in results_generation],
    "law_content": [r["law"].content for r in results_generation],
    "law_identifier": [r["law"].law_identifier for r in results_generation],
})

# Crear directorio de datos si no existe
os.makedirs("data", exist_ok=True)

# Guardar el dataset
dataset_filename = "data/queries_for_rag_ambiguous.csv"
df_evaluation.to_csv(dataset_filename, index=False)

print(f"💾 Dataset guardado en: {dataset_filename}")
print(f"📋 Dimensiones del dataset: {df_evaluation.shape}")

# Mostrar estadísticas básicas
print(f"\n📈 Estadísticas del dataset:")
print(f"   • Promedio caracteres por pregunta: {df_evaluation['query'].str.len().mean():.1f}")
print(f"   • Promedio caracteres por contenido legal: {df_evaluation['law_content'].str.len().mean():.1f}")
print(f"   • Número de leyes únicas: {df_evaluation['law_id'].nunique()}")
print(f"   • Número de artículos únicos: {df_evaluation['law_item_id'].nunique()}")

# Mostrar ejemplos de preguntas generadas
print(f"\n🔍 Ejemplos de preguntas generadas:")
sample_queries = df_evaluation['query'].sample(3)
for i, query in enumerate(sample_queries, 1):
    print(f"   {i}. {query}")


## 5. Análisis Descriptivo del Dataset

Análisis estadístico y visual del dataset generado para comprender sus características.


In [ ]:
# Cargar dataset (si ya existe)
try:
    df = pd.read_csv("data/queries_for_rag_ambiguous.csv")
    print(f"✅ Dataset cargado: {df.shape[0]} registros, {df.shape[1]} columnas")
except FileNotFoundError:
    df = df_evaluation
    print(f"✅ Usando dataset generado en memoria")


In [ ]:
# === ANÁLISIS DESCRIPTIVO COMPLETO ===
print("=" * 80)
print("📊 ANÁLISIS DESCRIPTIVO DEL DATASET LEGAL")
print("=" * 80)

print(f"\n📋 INFORMACIÓN BÁSICA:")
print(f"   • Número total de registros: {len(df):,}")
print(f"   • Número de columnas: {len(df.columns)}")
print(f"   • Memoria utilizada: {df.memory_usage(deep=True).sum() / 1024 ** 2:.2f} MB")

# Información detallada por columna
print(f"\n📊 DETALLE POR COLUMNA:")
for col in df.columns:
    null_count = df[col].isnull().sum()
    unique_count = df[col].nunique()
    dtype = df[col].dtype
    print(f"   • {col}: {unique_count:,} valores únicos, {null_count} nulos, tipo: {dtype}")


In [ ]:
# === ANÁLISIS ESTADÍSTICO DE LONGITUDES ===
query_lengths = df['query'].str.len()
content_lengths = df['law_content'].str.len()

print(f"\n📏 ANÁLISIS DE LONGITUDES:")
print(f"   📝 Preguntas:")
print(f"      - Promedio: {query_lengths.mean():.1f} caracteres")
print(f"      - Mediana: {query_lengths.median():.1f} caracteres")
print(f"      - Mín/Máx: {query_lengths.min()}/{query_lengths.max()} caracteres")
print(f"      - Desviación estándar: {query_lengths.std():.1f}")

print(f"   ⚖️ Contenido Legal:")
print(f"      - Promedio: {content_lengths.mean():.1f} caracteres")
print(f"      - Mediana: {content_lengths.median():.1f} caracteres")
print(f"      - Mín/Máx: {content_lengths.min()}/{content_lengths.max()} caracteres")
print(f"      - Desviación estándar: {content_lengths.std():.1f}")


In [ ]:
# === VISUALIZACIÓN COMPLETA ===
print("🎨 Generando visualizaciones...")

# Configurar la figura principal con subplots
fig = plt.figure(figsize=(20, 16))
fig.suptitle('Análisis Descriptivo del Dataset Legal - TFM', fontsize=16, fontweight='bold')

# 1. Distribución de leyes más frecuentes
ax1 = plt.subplot(3, 4, 1)
law_counts = df['law_id'].value_counts().head(15)
bars1 = ax1.bar(range(len(law_counts)), law_counts.values, color='#3498DB', alpha=0.8)
ax1.set_title('Top 15 Leyes por Frecuencia', fontweight='bold')
ax1.set_xlabel('Ranking de Ley')
ax1.set_ylabel('Número de Artículos')
ax1.grid(True, alpha=0.3)

# Añadir valores en las barras
for i, bar in enumerate(bars1):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2., height + 0.5,
             f'{int(height)}', ha='center', va='bottom', fontsize=8)

# 2. Distribución de longitud de queries
ax2 = plt.subplot(3, 4, 2)
ax2.hist(query_lengths, bins=30, color='#E74C3C', alpha=0.7, edgecolor='black')
ax2.set_title('Distribución Longitud de Preguntas', fontweight='bold')
ax2.set_xlabel('Caracteres')
ax2.set_ylabel('Frecuencia')
ax2.axvline(query_lengths.mean(), color='red', linestyle='--',
            label=f'Media: {query_lengths.mean():.0f}')
ax2.axvline(query_lengths.median(), color='orange', linestyle='--',
            label=f'Mediana: {query_lengths.median():.0f}')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# 3. Distribución de longitud de contenido legal
ax3 = plt.subplot(3, 4, 3)
ax3.hist(content_lengths, bins=30, color='#2ECC71', alpha=0.7, edgecolor='black')
ax3.set_title('Distribución Longitud Contenido Legal', fontweight='bold')
ax3.set_xlabel('Caracteres')
ax3.set_ylabel('Frecuencia')
ax3.axvline(content_lengths.mean(), color='red', linestyle='--',
            label=f'Media: {content_lengths.mean():.0f}')
ax3.axvline(content_lengths.median(), color='orange', linestyle='--',
            label=f'Mediana: {content_lengths.median():.0f}')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)

# 4. Top títulos de ley más frecuentes
ax4 = plt.subplot(3, 4, 4)
top_law_titles = df['law_title'].value_counts().head(10)
y_pos = np.arange(len(top_law_titles))
bars4 = ax4.barh(y_pos, top_law_titles.values, color='#9B59B6', alpha=0.8)
ax4.set_yticks(y_pos)
ax4.set_yticklabels([title[:30] + '...' if len(title) > 30 else title
                     for title in top_law_titles.index], fontsize=8)
ax4.set_title('Top 10 Títulos de Ley', fontweight='bold')
ax4.set_xlabel('Frecuencia')
ax4.grid(True, alpha=0.3)

# Añadir valores en las barras horizontales
for i, bar in enumerate(bars4):
    width = bar.get_width()
    ax4.text(width + 0.5, bar.get_y() + bar.get_height() / 2.,
             f'{int(width)}', ha='left', va='center', fontsize=8)

# 5. Distribución de artículos por ley
ax5 = plt.subplot(3, 4, 5)
articles_per_law = df.groupby('law_id')['law_item_id'].nunique()
ax5.hist(articles_per_law, bins=20, color='#F39C12', alpha=0.7, edgecolor='black')
ax5.set_title('Distribución: Artículos por Ley', fontweight='bold')
ax5.set_xlabel('Número de Artículos')
ax5.set_ylabel('Número de Leyes')
ax5.axvline(articles_per_law.mean(), color='red', linestyle='--',
            label=f'Media: {articles_per_law.mean():.1f}')
ax5.legend(fontsize=8)
ax5.grid(True, alpha=0.3)

# 6. Análisis de palabras frecuentes en queries
ax6 = plt.subplot(3, 4, 6)
# Limpiar y procesar texto
all_words = ' '.join(df['query'].str.lower()).split()
# Filtrar palabras comunes no informativas
stopwords = {'de', 'la', 'el', 'en', 'a', 'y', 'que', 'del', 'se', 'los', 'un', 'por', 'con', 'una', 'para', 'es', 'al',
             'lo', 'le', 'da', 'su', 'le'}
meaningful_words = [word for word in all_words if len(word) > 3 and word not in stopwords]
word_freq = Counter(meaningful_words)
common_words = dict(word_freq.most_common(15))

bars6 = ax6.bar(range(len(common_words)), list(common_words.values()),
                color='#1ABC9C', alpha=0.8)
ax6.set_title('Top 15 Palabras Relevantes en Preguntas', fontweight='bold')
ax6.set_xlabel('Palabras')
ax6.set_ylabel('Frecuencia')
ax6.set_xticks(range(len(common_words)))
ax6.set_xticklabels(list(common_words.keys()), rotation=45, ha='right', fontsize=8)
ax6.grid(True, alpha=0.3)

# 7. Mapa de calor - Correlaciones
ax7 = plt.subplot(3, 4, 7)
correlation_data = pd.DataFrame({
    'Long_Pregunta': query_lengths,
    'Long_Contenido': content_lengths,
    'ID_Ley': df['law_id'],
    'ID_Artículo': df['law_item_id']
})
corr_matrix = correlation_data.corr()
im = ax7.imshow(corr_matrix, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
ax7.set_title('Matriz de Correlaciones', fontweight='bold')
ax7.set_xticks(range(len(corr_matrix.columns)))
ax7.set_yticks(range(len(corr_matrix.columns)))
ax7.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=8)
ax7.set_yticklabels(corr_matrix.columns, fontsize=8)

# Añadir valores de correlación
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        ax7.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                 ha='center', va='center', fontsize=8,
                 color='white' if abs(corr_matrix.iloc[i, j]) > 0.5 else 'black')

# 8. Boxplot comparativo de longitudes
ax8 = plt.subplot(3, 4, 8)
length_data = [query_lengths, content_lengths]
labels = ['Preguntas', 'Contenido Legal']
bp = ax8.boxplot(length_data, labels=labels, patch_artist=True)
colors = ['#FF6B6B', '#4ECDC4']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax8.set_title('Comparación de Longitudes', fontweight='bold')
ax8.set_ylabel('Caracteres')
ax8.grid(True, alpha=0.3)

# 9. Distribución acumulativa
ax9 = plt.subplot(3, 4, 9)
sorted_query_lengths = np.sort(query_lengths)
sorted_content_lengths = np.sort(content_lengths)
cumulative_queries = np.arange(1, len(sorted_query_lengths) + 1) / len(sorted_query_lengths)
cumulative_content = np.arange(1, len(sorted_content_lengths) + 1) / len(sorted_content_lengths)

ax9.plot(sorted_query_lengths, cumulative_queries, color='#E74C3C', linewidth=2, label='Preguntas')
ax9.plot(sorted_content_lengths, cumulative_content, color='#2ECC71', linewidth=2, label='Contenido Legal')
ax9.set_title('Distribución Acumulativa', fontweight='bold')
ax9.set_xlabel('Longitud (caracteres)')
ax9.set_ylabel('Percentil')
ax9.legend(fontsize=8)
ax9.grid(True, alpha=0.3)

# 10. Top artículos más largos
ax10 = plt.subplot(3, 4, 10)
df_temp = df.copy()
df_temp['content_length'] = df_temp['law_content'].str.len()
top_long_df = df_temp.nlargest(10, 'content_length')
top_long_articles = top_long_df['law_item_title']
article_lengths = top_long_df['content_length']
y_pos = np.arange(len(top_long_articles))
bars10 = ax10.barh(y_pos, article_lengths, color='#D35400', alpha=0.8)
ax10.set_yticks(y_pos)
ax10.set_yticklabels([str(title)[:25] + '...' if len(str(title)) > 25 else str(title)
                      for title in top_long_articles], fontsize=8)
ax10.set_title('Top 10 Artículos más Largos', fontweight='bold')
ax10.set_xlabel('Caracteres')
ax10.grid(True, alpha=0.3)

# 11. Análisis temporal/secuencial
ax11 = plt.subplot(3, 4, 11)
sample_indices = range(0, min(1000, len(df)), 10)  # Muestra cada 10 elementos
sample_query_lengths = [query_lengths.iloc[i] for i in sample_indices]
sample_content_lengths = [content_lengths.iloc[i] for i in sample_indices]

ax11.scatter(sample_indices, sample_query_lengths, alpha=0.6, color='#FF6B6B', s=20, label='Preguntas')
ax11.scatter(sample_indices, sample_content_lengths, alpha=0.6, color='#4ECDC4', s=20, label='Contenido')
ax11.set_title('Dispersión de Longitudes', fontweight='bold')
ax11.set_xlabel('Índice de Muestra')
ax11.set_ylabel('Caracteres')
ax11.legend(fontsize=8)
ax11.grid(True, alpha=0.3)

# 12. Panel de estadísticas resumidas
ax12 = plt.subplot(3, 4, 12)
ax12.axis('off')

stats_text = f"""
ESTADÍSTICAS CLAVE DEL DATASET

📊 PREGUNTAS GENERADAS:
  • Total: {len(df):,} preguntas
  • Longitud media: {query_lengths.mean():.0f} caracteres
  • Longitud mediana: {query_lengths.median():.0f} caracteres
  • Rango: {query_lengths.min()}-{query_lengths.max()} caracteres
  • Palabras únicas: {len(set(meaningful_words)):,}

⚖️ CONTENIDO LEGAL:
  • Leyes únicas: {df['law_id'].nunique():,}
  • Artículos únicos: {df['law_item_id'].nunique():,}
  • Contenido medio: {content_lengths.mean():.0f} caracteres
  • Ley más representada: {law_counts.iloc[0]} artículos

📈 DISTRIBUCIÓN:
  • P25 preguntas: {query_lengths.quantile(0.25):.0f} chars
  • P75 preguntas: {query_lengths.quantile(0.75):.0f} chars
  • P95 contenido: {content_lengths.quantile(0.95):.0f} chars
"""

ax12.text(0.05, 0.95, stats_text, transform=ax12.transAxes, fontsize=9,
          verticalalignment='top', fontfamily='monospace',
          bbox=dict(boxstyle="round,pad=0.5", facecolor="lightblue", alpha=0.8))

plt.tight_layout()
plt.savefig('data/dataset_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualizaciones generadas y guardadas en 'data/dataset_analysis.png'")


In [ ]:
# === ANÁLISIS TEXTUAL DETALLADO ===
print(f"\n🔍 ANÁLISIS DETALLADO:")
print(f"   • Pregunta más corta: '{df.loc[query_lengths.idxmin(), 'query']}'")
print(f"   • Pregunta más larga: '{df.loc[query_lengths.idxmax(), 'query'][:100]}...'")
print(f"   • Ley con más artículos: {law_counts.index[0]} ({law_counts.iloc[0]} artículos)")
print(f"   • Percentil 90 longitud preguntas: {query_lengths.quantile(0.90):.0f} caracteres")
print(f"   • Percentil 90 longitud contenido: {content_lengths.quantile(0.90):.0f} caracteres")

# Análisis de diversidad léxica
unique_words_ratio = len(set(all_words)) / len(all_words) if all_words else 0
print(f"   • Diversidad léxica: {unique_words_ratio:.3f} (palabras únicas/total)")

print(f"\n💾 Dataset listo para evaluación del sistema RAG")


## 6. Evaluación del Sistema RAG

Evaluación sistemática del rendimiento del sistema RAG utilizando el dataset generado.


### 6.1 Configuración para Evaluación


In [ ]:
# Cargar el dataset de evaluación
print("📂 Cargando dataset para evaluación...")

try:
    df_eval = pd.read_csv("data/queries_for_rag_ambiguous.csv")
    print(f"✅ Dataset cargado: {df_eval.shape}")
except FileNotFoundError:
    print("⚠️ Dataset no encontrado, usando dataset generado en memoria")
    df_eval = df.copy()

# Configurar el sistema RAG para evaluación
print("🔧 Configurando sistema RAG para evaluación...")
rag_service_eval = RAGService(config=RAGConfig(
    model_name="gpt-5",
    agent_timeout=120,
    search_limit=20,
))

print(f"📊 Configuración de evaluación:")
print(f"   • Total de preguntas: {len(df_eval)}")
print(f"   • Modelo: gpt-5")
print(f"   • Límite de búsqueda: 20 resultados")
print(f"   • Timeout: 120 segundos")


### 6.2 Ejecución de la Evaluación Principal


In [ ]:
df_eval['law_id']

In [ ]:
# === EVALUACIÓN PRINCIPAL DEL SISTEMA RAG ===
print("🚀 INICIANDO EVALUACIÓN PRINCIPAL DEL SISTEMA RAG")
print("=" * 80)

# Métricas de evaluación
law_success = 0
law_item_success = 0
both_success = 0
evaluation_results = []
evaluation_errors = []

# Cargar datos de evaluación
try:
    df_eval = pd.read_csv("data/queries_for_rag_ambiguous.csv")
    print(f"✅ Dataset cargado: {len(df_eval)} preguntas")
except FileNotFoundError:
    print("❌ Error: Archivo de evaluación no encontrado")
    raise
except Exception as e:
    print(f"❌ Error cargando dataset: {str(e)}")
    raise

# Configurar servicio RAG
rag_service_eval = RAGService(config=RAGConfig(
    model_name="gpt-5",
    agent_timeout=120,
    search_limit=20,
))

# Configurar tamaño de muestra para evaluación
SAMPLE_SIZE = min(20, len(df_eval))  # Evaluar máximo 10 preguntas
print(f"📋 Evaluando {SAMPLE_SIZE} preguntas del dataset...")

# Iniciar evaluación
evaluation_start_time = datetime.now()

for i, (idx, row) in enumerate(df_eval.head(SAMPLE_SIZE).iterrows()):
    try:
        print(f"\n{'=' * 60}")
        print(f"🔍 Pregunta {i + 1}/{SAMPLE_SIZE}")
        print(f"{'=' * 60}")
        print(f"❓ Query: {row['query']}")

        # Procesar la consulta con el sistema RAG
        result, metadata = await rag_service_eval.process_query(row["query"])

        # Extraer IDs encontrados en las citas (con validación)
        law_ids_found = []
        law_item_ids_found = []

        if hasattr(result, 'citations') and result.citations:
            law_ids_found = [cite.law_unic_id for cite in result.citations
                           if hasattr(cite, 'law_unic_id') and cite.law_unic_id is not None]
            law_item_ids_found = [cite.law_item_id for cite in result.citations
                                if hasattr(cite, 'law_item_id') and cite.law_item_id is not None]

        # Verificar aciertos (manejar valores NaN)
        expected_law_id = row["law_id"] if not pd.isna(row["law_id"]) else None
        expected_law_item_id = row["law_item_id"] if not pd.isna(row["law_item_id"]) else None

        law_correct = expected_law_id in law_ids_found if expected_law_id is not None else False
        law_item_correct = expected_law_item_id in law_item_ids_found if expected_law_item_id is not None else False
        both_correct = law_correct and law_item_correct

        # Actualizar contadores
        if law_correct:
            law_success += 1
        if law_item_correct:
            law_item_success += 1
        if both_correct:
            both_success += 1

        # Guardar resultados detallados
        evaluation_results.append({
            'question_id': i,
            'original_index': idx,
            'query': row["query"],
            'expected_law_id': expected_law_id,
            'expected_law_item_id': expected_law_item_id,
            'found_law_ids': law_ids_found,
            'found_law_item_ids': law_item_ids_found,
            'law_correct': law_correct,
            'law_item_correct': law_item_correct,
            'both_correct': both_correct,
            'num_citations': len(result.citations) if hasattr(result, 'citations') and result.citations else 0,
            'response_length': len(result.response) if hasattr(result, 'response') and result.response else 0,
            'reasoning_length': len(result.reasoning) if hasattr(result, 'reasoning') and result.reasoning else 0,
        })

        # Mostrar resultado
        status_law = "✅" if law_correct else "❌"
        status_item = "✅" if law_item_correct else "❌"
        print(f"📊 Resultados:")
        print(f"   • Ley correcta: {status_law} ({expected_law_id} esperado)")
        print(f"   • Artículo correcto: {status_item} ({expected_law_item_id} esperado)")
        print(f"   • Citaciones encontradas: {len(result.citations) if hasattr(result, 'citations') and result.citations else 0}")
        print(f"   • IDs de leyes encontrados: {law_ids_found}")
        print(f"   • IDs de artículos encontrados: {law_item_ids_found}")

        # Mostrar progreso acumulado cada 5 preguntas
        if (i + 1) % 5 == 0 or i == SAMPLE_SIZE - 1:
            current_law_acc = (law_success / (i + 1)) * 100
            current_item_acc = (law_item_success / (i + 1)) * 100
            current_both_acc = (both_success / (i + 1)) * 100
            elapsed_time = (datetime.now() - evaluation_start_time).total_seconds()

            print(f"\n📈 PROGRESO ACUMULADO (después de {i + 1} preguntas):")
            print(f"   • Precisión Ley: {current_law_acc:.1f}%")
            print(f"   • Precisión Artículo: {current_item_acc:.1f}%")
            print(f"   • Precisión Ambos: {current_both_acc:.1f}%")
            print(f"   • Tiempo transcurrido: {elapsed_time:.1f} segundos")
            print(f"   • Tiempo promedio por pregunta: {elapsed_time / (i + 1):.1f}s")

    except Exception as e:
        error_info = {
            'question_id': i,
            'query': row["query"],
            'error': str(e),
            'timestamp': datetime.now()
        }
        evaluation_errors.append(error_info)
        print(f"❌ Error procesando pregunta {i + 1}: {str(e)}")
        continue

# Finalizar evaluación
evaluation_end_time = datetime.now()
total_evaluation_time = (evaluation_end_time - evaluation_start_time).total_seconds()

print(f"\n🎉 EVALUACIÓN COMPLETADA")
print(f"=" * 80)
print(f"⏱️ Tiempo total: {total_evaluation_time:.1f} segundos")
print(f"📊 Preguntas procesadas: {len(evaluation_results)}")
print(f"❌ Errores encontrados: {len(evaluation_errors)}")

# Calcular y mostrar métricas finales
if len(evaluation_results) > 0:
    final_law_acc = (law_success / len(evaluation_results)) * 100
    final_item_acc = (law_item_success / len(evaluation_results)) * 100
    final_both_acc = (both_success / len(evaluation_results)) * 100

    print(f"\n📊 MÉTRICAS FINALES:")
    print(f"   • Precisión Ley: {final_law_acc:.2f}% ({law_success}/{len(evaluation_results)})")
    print(f"   • Precisión Artículo: {final_item_acc:.2f}% ({law_item_success}/{len(evaluation_results)})")
    print(f"   • Precisión Ambos: {final_both_acc:.2f}% ({both_success}/{len(evaluation_results)})")
    print(f"   • Tiempo promedio por consulta: {total_evaluation_time / len(evaluation_results):.2f}s")

# Guardar resultados
if evaluation_results:
    results_df = pd.DataFrame(evaluation_results)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_filename = f"evaluation_results_{timestamp}.csv"
    results_df.to_csv(results_filename, index=False)
    print(f"💾 Resultados guardados en: {results_filename}")

# Mostrar errores si los hay
if evaluation_errors:
    print(f"\n⚠️ Se encontraron {len(evaluation_errors)} errores:")
    for error in evaluation_errors:
        print(f"   • Pregunta {error['question_id']}: {error['error']}")

print(f"\n✨ Evaluación completada exitosamente")

### 6.3 Guardado de Resultados de Evaluación


In [ ]:
# === GUARDAR RESULTADOS DE LA EVALUACIÓN ===
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Crear estructura de datos completa para guardar
evaluation_data = {
    'metadata': {
        'timestamp': timestamp,
        'total_questions': len(evaluation_results),
        'sample_size': SAMPLE_SIZE,
        'model_name': "gpt-4",
        'search_limit': 20,
        'evaluation_time_seconds': total_evaluation_time,
        'avg_time_per_question': total_evaluation_time / len(evaluation_results) if evaluation_results else 0
    },
    'metrics': {
        'law_success': law_success,
        'law_item_success': law_item_success,
        'both_success': both_success,
        'law_accuracy': (law_success / len(evaluation_results)) * 100 if evaluation_results else 0,
        'law_item_accuracy': (law_item_success / len(evaluation_results)) * 100 if evaluation_results else 0,
        'both_accuracy': (both_success / len(evaluation_results)) * 100 if evaluation_results else 0,
        'total_errors': len(evaluation_errors)
    },
    'detailed_results': evaluation_results,
    'errors': evaluation_errors
}

csv_filename = f"data/evaluation_results_{timestamp}.csv"
pickle_filename = f"data/evaluation_results_{timestamp}.pkl"
df_results = pd.DataFrame(evaluation_results)
df_results.to_csv(csv_filename, index=False)

print(f"💾 RESULTADOS GUARDADOS:")
print(f"   • Archivo completo (pickle): {pickle_filename}")
print(f"   • Resultados tabulares (CSV): {csv_filename}")
print(f"   • Tamaño del archivo pickle: {os.path.getsize(pickle_filename) / 1024:.2f} KB")

# Calcular métricas finales
total_evaluated = len(evaluation_results)
law_accuracy = (law_success / total_evaluated) * 100 if total_evaluated > 0 else 0
item_accuracy = (law_item_success / total_evaluated) * 100 if total_evaluated > 0 else 0
both_accuracy = (both_success / total_evaluated) * 100 if total_evaluated > 0 else 0

print(f"\n📊 MÉTRICAS FINALES:")
print(f"   • Total evaluado: {total_evaluated} preguntas")
print(f"   • Precisión Ley: {law_success}/{total_evaluated} ({law_accuracy:.1f}%)")
print(f"   • Precisión Artículo: {law_item_success}/{total_evaluated} ({item_accuracy:.1f}%)")
print(f"   • Precisión Ambos: {both_success}/{total_evaluated} ({both_accuracy:.1f}%)")
print(f"   • Tasa de error: {len(evaluation_errors) / SAMPLE_SIZE * 100:.1f}%")


## 7. Análisis de Resultados

Análisis detallado de los resultados obtenidos en la evaluación del sistema RAG.


In [ ]:
# === ANÁLISIS VISUAL DE RESULTADOS ===
print("🎨 Generando análisis visual de resultados...")


# Crear figura para análisis de resultados
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle(f'Análisis de Resultados - Evaluación RAG ({timestamp})', fontsize=16, fontweight='bold')

# 1. Gráfico de barras con precisión por métrica
metrics = ['Ley Correcta', 'Artículo Correcto', 'Ambos Correctos']
accuracies = [law_accuracy, item_accuracy, both_accuracy]
colors = ['#3498DB', '#E74C3C', '#2ECC71']

bars = ax1.bar(metrics, accuracies, color=colors, alpha=0.8, edgecolor='black')
ax1.set_title('Precisión por Métrica (%)', fontweight='bold')
ax1.set_ylim(0, 100)
ax1.set_ylabel('Precisión (%)')
ax1.grid(True, alpha=0.3)

# Añadir valores en las barras
for bar, acc in zip(bars, accuracies):
    ax1.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 2,
             f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')

# Añadir línea de objetivo (por ejemplo, 70%)
ax1.axhline(y=70, color='red', linestyle=':', alpha=0.7, label='Objetivo 70%')
ax1.legend()

# 2. Distribución de número de citaciones
if evaluation_results:
    citations_counts = [r['num_citations'] for r in evaluation_results]
    ax2.hist(citations_counts, bins=max(1, len(set(citations_counts))),
             alpha=0.7, color='#F39C12', edgecolor='black')
    ax2.set_title('Distribución de Citaciones por Pregunta', fontweight='bold')
    ax2.set_xlabel('Número de Citaciones')
    ax2.set_ylabel('Frecuencia')
    ax2.axvline(np.mean(citations_counts), color='red', linestyle='--',
                label=f'Media: {np.mean(citations_counts):.1f}')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

# 3. Gráfico de pizza con distribución de resultados
only_law = law_success - both_success
only_item = law_item_success - both_success
neither = total_evaluated - law_success - law_item_success + both_success

sizes = [both_success, only_law, only_item, neither]
labels = ['Ambos Correctos', 'Solo Ley', 'Solo Artículo', 'Ninguno Correcto']
colors_pie = ['#2ECC71', '#F39C12', '#E74C3C', '#95A5A6']
explode = (0.1, 0, 0, 0)  # Destacar "Ambos Correctos"

wedges, texts, autotexts = ax3.pie(sizes, labels=labels, colors=colors_pie,
                                   autopct='%1.1f%%', startangle=90, explode=explode)
ax3.set_title('Distribución de Resultados', fontweight='bold')

# Mejorar el formato de los porcentajes
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

# 4. Tendencia de precisión a lo largo de las preguntas
if evaluation_results:
    correct_trend = [r['both_correct'] for r in evaluation_results]
    question_numbers = range(1, len(correct_trend) + 1)

    # Calcular media móvil para suavizar la tendencia
    window_size = min(10, len(correct_trend) // 5)
    if window_size > 1:
        moving_avg = pd.Series(correct_trend).rolling(window=window_size, center=True).mean()
        ax4.plot(question_numbers, moving_avg, color='red', linewidth=2,
                 label=f'Media móvil (ventana={window_size})')

    ax4.scatter(question_numbers, correct_trend, alpha=0.6, color='#9B59B6', s=30)
    ax4.set_title('Evolución de Precisión por Pregunta', fontweight='bold')
    ax4.set_xlabel('Número de Pregunta')
    ax4.set_ylabel('Ambos Correctos (1=Sí, 0=No)')
    ax4.set_ylim(-0.1, 1.1)
    ax4.grid(True, alpha=0.3)

    if window_size > 1:
        ax4.legend()

    # Añadir línea de tendencia general
    from numpy.polynomial.polynomial import Polynomial

    if len(question_numbers) > 5:
        p = Polynomial.fit(question_numbers, correct_trend, 1)
        trend_line = p(question_numbers)
        ax4.plot(question_numbers, trend_line, '--', color='orange', alpha=0.7,
                 label='Tendencia lineal')
        ax4.legend()

plt.tight_layout()
plt.savefig(f'data/evaluation_results_{timestamp}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Gráficos guardados en: data/evaluation_results_{timestamp}.png")


In [ ]:
# === ANÁLISIS DETALLADO DE CASOS ===
print(f"\n🔍 ANÁLISIS DETALLADO DE CASOS:")
print(f"=" * 80)

# Casos exitosos
successful_cases = [r for r in evaluation_results if r['both_correct']]
print(f"✅ CASOS COMPLETAMENTE EXITOSOS ({len(successful_cases)} casos):")
if successful_cases:
    for case in successful_cases[:3]:  # Mostrar primeros 3
        print(f"   • Q{case['question_id']}: {case['query'][:80]}...")
        print(f"     → {case['num_citations']} citaciones encontradas")

# Casos parcialmente exitosos (solo ley)
partial_law_cases = [r for r in evaluation_results if r['law_correct'] and not r['law_item_correct']]
print(f"\n⚠️ CASOS CON SOLO LEY CORRECTA ({len(partial_law_cases)} casos):")
if partial_law_cases:
    for case in partial_law_cases[:3]:
        print(f"   • Q{case['question_id']}: {case['query'][:80]}...")
        print(f"     → Esperado: Art. {case['expected_law_item_id']}, Encontrado: {case['found_law_item_ids']}")

# Casos parcialmente exitosos (solo artículo)
partial_item_cases = [r for r in evaluation_results if not r['law_correct'] and r['law_item_correct']]
print(f"\n⚠️ CASOS CON SOLO ARTÍCULO CORRECTO ({len(partial_item_cases)} casos):")
if partial_item_cases:
    for case in partial_item_cases[:3]:
        print(f"   • Q{case['question_id']}: {case['query'][:80]}...")
        print(f"     → Esperado: Ley {case['expected_law_id']}, Encontrado: {case['found_law_ids']}")

# Casos completamente fallidos
failed_cases = [r for r in evaluation_results if not r['law_correct'] and not r['law_item_correct']]
print(f"\n❌ CASOS COMPLETAMENTE FALLIDOS ({len(failed_cases)} casos):")
if failed_cases:
    for case in failed_cases[:3]:
        print(f"   • Q{case['question_id']}: {case['query'][:80]}...")
        print(f"     → Esperado: Ley {case['expected_law_id']}, Art. {case['expected_law_item_id']}")
        print(f"     → Encontrado: Leyes {case['found_law_ids']}, Arts. {case['found_law_item_ids']}")


In [ ]:
# === ANÁLISIS DE CORRELACIONES Y PATRONES ===
print(f"\n📊 ANÁLISIS DE CORRELACIONES:")
print(f"=" * 60)

if evaluation_results:
    df_analysis = pd.DataFrame(evaluation_results)

    # Análisis de correlación entre número de citaciones y éxito
    citations_vs_success = df_analysis.groupby('num_citations').agg({
        'both_correct': 'mean',
        'law_correct': 'mean',
        'law_item_correct': 'mean'
    }).round(3)

    print("📈 Correlación Citaciones vs Éxito:")
    print(citations_vs_success.head())

    # Análisis de longitud de query vs éxito
    df_analysis['query_length'] = df_analysis['query'].str.len()
    length_bins = pd.cut(df_analysis['query_length'], bins=5,
                         labels=['Muy Corta', 'Corta', 'Media', 'Larga', 'Muy Larga'])
    length_vs_success = df_analysis.groupby(length_bins).agg({
        'both_correct': 'mean',
        'num_citations': 'mean'
    }).round(3)

    print(f"\n📏 Correlación Longitud Query vs Éxito:")
    print(length_vs_success)

    # Estadísticas de tiempo de procesamiento
    if 'processing_time' in df_analysis.columns:
        avg_processing_time = df_analysis['processing_time'].mean()
        print(f"\n⏱️ Tiempo promedio de procesamiento: {avg_processing_time:.2f} segundos")

    # Análisis de diversidad en las respuestas
    unique_law_ids_found = len(set([lid for sublist in df_analysis['found_law_ids'] for lid in sublist]))
    unique_expected_law_ids = len(set(df_analysis['expected_law_id']))

    print(f"\n🎯 Diversidad de Respuestas:")
    print(f"   • Leyes únicas esperadas: {unique_expected_law_ids}")
    print(f"   • Leyes únicas encontradas: {unique_law_ids_found}")
    print(f"   • Cobertura de leyes: {(unique_law_ids_found / unique_expected_law_ids) * 100:.1f}%")


### 7.1 Análisis de Caso de Fallo Detallado


In [ ]:
# === ANÁLISIS DETALLADO DE UN CASO FALLIDO ===
print("🔬 ANÁLISIS DETALLADO DE CASO FALLIDO")
print("=" * 80)

if failed_cases:
    # Seleccionar un caso fallido para análisis detallado
    case_example = failed_cases[0]

    print(f"📋 CASO SELECCIONADO:")
    print(f"   • ID: Q{case_example['question_id']}")
    print(f"   • Query: {case_example['query']}")
    print(f"   • Ley esperada: {case_example['expected_law_id']}")
    print(f"   • Artículo esperado: {case_example['expected_law_item_id']}")
    print(f"   • Leyes encontradas: {case_example['found_law_ids']}")
    print(f"   • Artículos encontrados: {case_example['found_law_item_ids']}")
    print(f"   • Número de citaciones: {case_example['num_citations']}")

    # Reejecutar la consulta para obtener más detalles
    print(f"\n🔄 RE-EJECUTANDO CONSULTA PARA ANÁLISIS DETALLADO...")
    detailed_result, detailed_metadata = await rag_service_eval.process_query(case_example['query'])

    print(f"\n📝 RESPUESTA GENERADA:")
    print(f"{detailed_result.response[:500]}...")

    print(f"\n🧠 RAZONAMIENTO:")
    if detailed_result.reasoning:
        print(f"{detailed_result.reasoning[:500]}...")
    else:
        print("No hay razonamiento disponible")

    print(f"\n📚 CITACIONES DETALLADAS:")
    for i, cite in enumerate(detailed_result.citations, 1):
        print(f"   {i}. {cite}")

    # Análisis de por qué pudo haber fallado
    print(f"\n🔍 POSIBLES CAUSAS DEL FALLO:")
    if not detailed_result.citations:
        print("   • No se encontraron citaciones relevantes")
    elif case_example['num_citations'] > 5:
        print("   • Demasiadas citaciones - posible ruido en los resultados")
    elif len(case_example['query']) < 50:
        print("   • Query muy corta - podría ser demasiado ambigua")
    elif len(case_example['query']) > 200:
        print("   • Query muy larga - podría confundir al sistema")
    else:
        print("   • Posible problema de coincidencia semántica")


## 8. Optimización de Parámetros

Evaluación del sistema con diferentes configuraciones para encontrar los parámetros óptimos.


### 8.1 Evaluación con Límite de Búsqueda Reducido


In [ ]:
# === EVALUACIÓN CON PARÁMETROS OPTIMIZADOS ===
print("🔧 EVALUACIÓN CON LÍMITE DE BÚSQUEDA REDUCIDO")
print("=" * 80)

# Configurar sistema RAG con límite reducido
rag_service_optimized = RAGService(config=RAGConfig(
    model_name="gpt-4",
    agent_timeout=120,
    search_limit=15,  # Reducido de 20 a 15
))

print(f"⚙️ Configuración optimizada:")
print(f"   • Límite de búsqueda: 15 (reducido de 20)")
print(f"   • Objetivo: Reducir ruido y mejorar precisión")

# Métricas para evaluación optimizada
law_success_opt = 0
law_item_success_opt = 0
both_success_opt = 0
results_optimized = []

# Evaluar con muestra más pequeña para optimización
SAMPLE_SIZE_OPT = min(50, len(df_eval))
print(f"📊 Evaluando {SAMPLE_SIZE_OPT} preguntas con configuración optimizada...")

start_time_opt = datetime.now()

for i, (idx, row) in enumerate(df_eval.head(SAMPLE_SIZE_OPT).iterrows()):
    try:
        print(f"🔍 Pregunta optimizada {i + 1}/{SAMPLE_SIZE_OPT}: {row['query'][:60]}...")

        result_opt, metadata_opt = await rag_service_optimized.process_query(row["query"])

        law_ids_found_opt = [cite.law_unic_id for cite in result_opt.citations]
        law_item_ids_found_opt = [cite.law_item_id for cite in result_opt.citations]

        law_correct_opt = row["law_id"] in law_ids_found_opt
        law_item_correct_opt = row["law_item_id"] in law_item_ids_found_opt
        both_correct_opt = law_correct_opt and law_item_correct_opt

        if law_correct_opt:
            law_success_opt += 1
        if law_item_correct_opt:
            law_item_success_opt += 1
        if both_correct_opt:
            both_success_opt += 1

        results_optimized.append({
            'question_id': i,
            'query': row["query"],
            'expected_law_id': row["law_id"],
            'expected_law_item_id': row["law_item_id"],
            'found_law_ids': law_ids_found_opt,
            'found_law_item_ids': law_item_ids_found_opt,
            'law_correct': law_correct_opt,
            'law_item_correct': law_item_correct_opt,
            'both_correct': both_correct_opt,
            'num_citations': len(result_opt.citations)
        })

        status_law_opt = "✅" if law_correct_opt else "❌"
        status_item_opt = "✅" if law_item_correct_opt else "❌"
        print(f"   Ley: {status_law_opt}, Artículo: {status_item_opt}, Citas: {len(result_opt.citations)}")

    except Exception as e:
        print(f"❌ Error en pregunta optimizada {i + 1}: {str(e)}")
        continue

end_time_opt = datetime.now()
total_time_opt = (end_time_opt - start_time_opt).total_seconds()

# Calcular métricas optimizadas
total_opt = len(results_optimized)
law_accuracy_opt = (law_success_opt / total_opt) * 100 if total_opt > 0 else 0
item_accuracy_opt = (law_item_success_opt / total_opt) * 100 if total_opt > 0 else 0
both_accuracy_opt = (both_success_opt / total_opt) * 100 if total_opt > 0 else 0

print(f"\n📊 RESULTADOS OPTIMIZACIÓN:")
print(f"   • Tiempo total: {total_time_opt:.1f}s")
print(f"   • Ley correcta: {law_success_opt}/{total_opt} ({law_accuracy_opt:.1f}%)")
print(f"   • Artículo correcto: {law_item_success_opt}/{total_opt} ({item_accuracy_opt:.1f}%)")
print(f"   • Ambos correctos: {both_success_opt}/{total_opt} ({both_accuracy_opt:.1f}%)")


### 8.2 Comparación de Configuraciones


In [ ]:
# === COMPARACIÓN DE CONFIGURACIONES ===
print("📈 COMPARACIÓN DE CONFIGURACIONES")
print("=" * 80)

# Crear tabla comparativa
comparison_data = {
    'Métrica': ['Precisión Ley (%)', 'Precisión Artículo (%)', 'Precisión Ambos (%)',
                'Citaciones Promedio', 'Tiempo/Pregunta (s)'],
    'Configuración Original': [
        f"{law_accuracy:.1f}",
        f"{item_accuracy:.1f}",
        f"{both_accuracy:.1f}",
        f"{np.mean([r['num_citations'] for r in evaluation_results]):.1f}" if evaluation_results else "N/A",
        f"{total_evaluation_time / len(evaluation_results):.1f}" if evaluation_results else "N/A"
    ],
    'Configuración Optimizada': [
        f"{law_accuracy_opt:.1f}",
        f"{item_accuracy_opt:.1f}",
        f"{both_accuracy_opt:.1f}",
        f"{np.mean([r['num_citations'] for r in results_optimized]):.1f}" if results_optimized else "N/A",
        f"{total_time_opt / len(results_optimized):.1f}" if results_optimized else "N/A"
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

# Análisis de mejora
if results_optimized and evaluation_results:
    improvement_both = both_accuracy_opt - both_accuracy
    improvement_law = law_accuracy_opt - law_accuracy
    improvement_item = item_accuracy_opt - item_accuracy

    print(f"\n📊 ANÁLISIS DE MEJORA:")
    print(f"   • Mejora en precisión ambos: {improvement_both:+.1f} puntos porcentuales")
    print(f"   • Mejora en precisión ley: {improvement_law:+.1f} puntos porcentuales")
    print(f"   • Mejora en precisión artículo: {improvement_item:+.1f} puntos porcentuales")

    if improvement_both > 0:
        print(f"   ✅ La configuración optimizada muestra mejora")
    else:
        print(f"   ⚠️ La configuración optimizada no mejora significativamente")


In [ ]:
# === VISUALIZACIÓN COMPARATIVA ===
print("🎨 Generando visualización comparativa...")

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Comparación de Configuraciones - Sistema RAG', fontsize=16, fontweight='bold')

# 1. Comparación de precisión por métrica
metrics_comp = ['Ley', 'Artículo', 'Ambos']
original_scores = [law_accuracy, item_accuracy, both_accuracy]
optimized_scores = [law_accuracy_opt, item_accuracy_opt, both_accuracy_opt]

x = np.arange(len(metrics_comp))
width = 0.35

bars1 = ax1.bar(x - width / 2, original_scores, width, label='Original (límite=20)',
                color='#3498DB', alpha=0.8)
bars2 = ax1.bar(x + width / 2, optimized_scores, width, label='Optimizada (límite=15)',
                color='#E74C3C', alpha=0.8)

ax1.set_title('Comparación de Precisión', fontweight='bold')
ax1.set_ylabel('Precisión (%)')
ax1.set_xlabel('Métricas')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics_comp)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Añadir valores en las barras
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width() / 2., height + 1,
                 f'{height:.1f}%', ha='center', va='bottom', fontsize=9)

# 2. Distribución de citaciones - comparación
if evaluation_results and results_optimized:
    citations_orig = [r['num_citations'] for r in evaluation_results[:len(results_optimized)]]
    citations_opt = [r['num_citations'] for r in results_optimized]

    ax2.hist(citations_orig, bins=range(0, max(max(citations_orig), max(citations_opt)) + 2),
             alpha=0.6, color='#3498DB', label='Original', density=True)
    ax2.hist(citations_opt, bins=range(0, max(max(citations_orig), max(citations_opt)) + 2),
             alpha=0.6, color='#E74C3C', label='Optimizada', density=True)

    ax2.set_title('Distribución de Citaciones', fontweight='bold')
    ax2.set_xlabel('Número de Citaciones')
    ax2.set_ylabel('Densidad')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

# 3. Matriz de confusión simplificada para configuración original
if evaluation_results:
    conf_data_orig = {
        'Ambos': both_success,
        'Solo Ley': law_success - both_success,
        'Solo Artículo': law_item_success - both_success,
        'Ninguno': len(evaluation_results) - law_success - law_item_success + both_success
    }

    labels_conf = list(conf_data_orig.keys())
    sizes_conf = list(conf_data_orig.values())
    colors_conf = ['#2ECC71', '#F39C12', '#E74C3C', '#95A5A6']

    ax3.pie(sizes_conf, labels=labels_conf, colors=colors_conf, autopct='%1.1f%%',
            startangle=90)
    ax3.set_title('Distribución Original', fontweight='bold')

# 4. Matriz de confusión simplificada para configuración optimizada
if results_optimized:
    conf_data_opt = {
        'Ambos': both_success_opt,
        'Solo Ley': law_success_opt - both_success_opt,
        'Solo Artículo': law_item_success_opt - both_success_opt,
        'Ninguno': len(results_optimized) - law_success_opt - law_item_success_opt + both_success_opt
    }

    labels_conf_opt = list(conf_data_opt.keys())
    sizes_conf_opt = list(conf_data_opt.values())

    ax4.pie(sizes_conf_opt, labels=labels_conf_opt, colors=colors_conf, autopct='%1.1f%%',
            startangle=90)
    ax4.set_title('Distribución Optimizada', fontweight='bold')

plt.tight_layout()
plt.savefig(f'data/configuration_comparison_{timestamp}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Comparación visual guardada en: data/configuration_comparison_{timestamp}.png")


### 8.3 Generación de Preguntas Simplificadas


In [ ]:
# === GENERACIÓN DE DATASET CON PREGUNTAS SIMPLIFICADAS ===
print("🎯 GENERANDO DATASET CON PREGUNTAS SIMPLIFICADAS")
print("=" * 80)

print("💡 Hipótesis: Preguntas más directas pueden mejorar la precisión del sistema")

# Generar preguntas más directas y específicas
openai_client_simple = OpenAIRepository()
results_simple_generation = []
errors_simple = 0

# Usar una muestra de las leyes originales
laws_for_simple = df_eval.head(50)  # Muestra más pequeña para prueba
print(f"📚 Generando preguntas simplificadas para {len(laws_for_simple)} casos...")

for i, (idx, law_row) in enumerate(laws_for_simple.iterrows()):
    try:
        # Sistema prompt optimizado para preguntas más directas
        system_prompt_simple = (
            "Eres un estudiante de la Policía Nacional que formula consultas "
            "para poner a prueba un sistema RAG. Genera una sola pregunta "
            "o caso práctico breve basado en el texto legal. "
            "No puedes hacer menciones literales a la ley ni al artículo, pero sí nombrar conceptos legales, "
            "incluso el título de la ley. Puedes pedir un artículo concreto. "
            "Hazlo en menos de dos frases, de forma natural y directa."
        )

        response_simple = openai_client_simple.generate_text(
            system=system_prompt_simple,
            prompt=f"Texto legal: {law_row['law_content']}"
        )

        results_simple_generation.append({
            "law_data": law_row,
            "simplified_query": response_simple
        })

        if (i + 1) % 10 == 0:
            print(f"   ✅ Procesadas {i + 1}/{len(laws_for_simple)} leyes...")

    except Exception as e:
        errors_simple += 1
        print(f"   ⚠️ Error en ley {i + 1}: {str(e)}")
        continue

print(f"\n🎉 Generación de preguntas simplificadas completada!")
print(f"   • Preguntas generadas: {len(results_simple_generation)}")
print(f"   • Errores: {errors_simple}")

# Crear DataFrame simplificado
df_simple = pd.DataFrame({
    "query": [r["simplified_query"] for r in results_simple_generation],
    "law_id": [r["law_data"]["law_id"] for r in results_simple_generation],
    "law_title": [r["law_data"]["law_title"] for r in results_simple_generation],
    "law_item_id": [r["law_data"]["law_item_id"] for r in results_simple_generation],
    "law_item_title": [r["law_data"]["law_item_title"] for r in results_simple_generation],
    "law_content": [r["law_data"]["law_content"] for r in results_simple_generation],
    "law_identifier": [r["law_data"]["law_identifier"] for r in results_simple_generation],
})

# Guardar dataset simplificado
simple_filename = "data/queries_simplified.csv"
df_simple.to_csv(simple_filename, index=False)
print(f"💾 Dataset simplificado guardado en: {simple_filename}")

# Mostrar ejemplos de preguntas simplificadas
print(f"\n🔍 Ejemplos de preguntas simplificadas:")
for i, query in enumerate(df_simple['query'].head(3), 1):
    print(f"   {i}. {query}")


### 8.4 Evaluación con Preguntas Simplificadas


In [ ]:
# === EVALUACIÓN CON PREGUNTAS SIMPLIFICADAS ===
print("🚀 EVALUACIÓN CON PREGUNTAS SIMPLIFICADAS")
print("=" * 80)

# Configurar sistema RAG para preguntas simplificadas
rag_service_simple = RAGService(config=RAGConfig(
    model_name="gpt-4",
    agent_timeout=120,
    search_limit=15,  # Usar configuración optimizada
))

# Métricas para evaluación simplificada
law_success_simple = 0
law_item_success_simple = 0
both_success_simple = 0
results_simplified = []

print(f"📊 Evaluando {len(df_simple)} preguntas simplificadas...")
start_time_simple = datetime.now()

for i, row in df_simple.iterrows():
    try:
        print(f"🔍 Pregunta simplificada {i + 1}/{len(df_simple)}: {row['query'][:60]}...")

        result_simple, metadata_simple = await rag_service_simple.process_query(row["query"])

        law_ids_found_simple = [cite.law_unic_id for cite in result_simple.citations]
        law_item_ids_found_simple = [cite.law_item_id for cite in result_simple.citations]

        law_correct_simple = row["law_id"] in law_ids_found_simple
        law_item_correct_simple = row["law_item_id"] in law_item_ids_found_simple
        both_correct_simple = law_correct_simple and law_item_correct_simple

        if law_correct_simple:
            law_success_simple += 1
        if law_item_correct_simple:
            law_item_success_simple += 1
        if both_correct_simple:
            both_success_simple += 1

        results_simplified.append({
            'question_id': i,
            'query': row["query"],
            'expected_law_id': row["law_id"],
            'expected_law_item_id': row["law_item_id"],
            'found_law_ids': law_ids_found_simple,
            'found_law_item_ids': law_item_ids_found_simple,
            'law_correct': law_correct_simple,
            'law_item_correct': law_item_correct_simple,
            'both_correct': both_correct_simple,
            'num_citations': len(result_simple.citations)
        })

        status_law_simple = "✅" if law_correct_simple else "❌"
        status_item_simple = "✅" if law_item_correct_simple else "❌"
        print(f"   Ley: {status_law_simple}, Artículo: {status_item_simple}, Citas: {len(result_simple.citations)}")

        # Mostrar progreso cada 10 preguntas
        if (i + 1) % 10 == 0:
            current_both_acc = (both_success_simple / (i + 1)) * 100
            print(f"   📈 Precisión acumulada (ambos): {current_both_acc:.1f}%")

    except Exception as e:
        print(f"❌ Error en pregunta simplificada {i + 1}: {str(e)}")
        continue

end_time_simple = datetime.now()
total_time_simple = (end_time_simple - start_time_simple).total_seconds()

# Calcular métricas finales para preguntas simplificadas
total_simple = len(results_simplified)
law_accuracy_simple = (law_success_simple / total_simple) * 100 if total_simple > 0 else 0
item_accuracy_simple = (law_item_success_simple / total_simple) * 100 if total_simple > 0 else 0
both_accuracy_simple = (both_success_simple / total_simple) * 100 if total_simple > 0 else 0

print(f"\n📊 RESULTADOS CON PREGUNTAS SIMPLIFICADAS:")
print(f"   • Tiempo total: {total_time_simple:.1f}s ({total_time_simple / total_simple:.1f}s/pregunta)")
print(f"   • Ley correcta: {law_success_simple}/{total_simple} ({law_accuracy_simple:.1f}%)")
print(f"   • Artículo correcto: {law_item_success_simple}/{total_simple} ({item_accuracy_simple:.1f}%)")
print(f"   • Ambos correctos: {both_success_simple}/{total_simple} ({both_accuracy_simple:.1f}%)")

# Guardar resultados de preguntas simplificadas
timestamp_simple = datetime.now().strftime("%Y%m%d_%H%M%S")
simple_results_data = {
    'metadata': {
        'timestamp': timestamp_simple,
        'total_questions': total_simple,
        'evaluation_time_seconds': total_time_simple,
        'configuration': 'simplified_queries_limit_15'
    },
    'metrics': {
        'law_accuracy': law_accuracy_simple,
        'law_item_accuracy': item_accuracy_simple,
        'both_accuracy': both_accuracy_simple
    },
    'detailed_results': results_simplified
}

with open(f'data/rag_evaluation_simplified_{timestamp_simple}.pkl', 'wb') as f:
    pickle.dump(simple_results_data, f)

print(f"💾 Resultados guardados en: data/rag_evaluation_simplified_{timestamp_simple}.pkl")


## 9. Análisis Final y Conclusiones

Síntesis de todos los resultados obtenidos y conclusiones del estudio.


In [ ]:
# === RESUMEN COMPARATIVO FINAL ===
print("📊 RESUMEN COMPARATIVO FINAL - TODAS LAS CONFIGURACIONES")
print("=" * 100)

# Crear tabla comparativa completa
final_comparison = {
    'Configuración': [
        'Original (límite=20)',
        'Optimizada (límite=15)',
        'Preguntas Simplificadas (límite=15)'
    ],
    'Precisión Ley (%)': [
        f"{law_accuracy:.1f}" if 'law_accuracy' in locals() else "N/A",
        f"{law_accuracy_opt:.1f}" if 'law_accuracy_opt' in locals() else "N/A",
        f"{law_accuracy_simple:.1f}" if 'law_accuracy_simple' in locals() else "N/A"
    ],
    'Precisión Artículo (%)': [
        f"{item_accuracy:.1f}" if 'item_accuracy' in locals() else "N/A",
        f"{item_accuracy_opt:.1f}" if 'item_accuracy_opt' in locals() else "N/A",
        f"{item_accuracy_simple:.1f}" if 'item_accuracy_simple' in locals() else "N/A"
    ],
    'Precisión Ambos (%)': [
        f"{both_accuracy:.1f}" if 'both_accuracy' in locals() else "N/A",
        f"{both_accuracy_opt:.1f}" if 'both_accuracy_opt' in locals() else "N/A",
        f"{both_accuracy_simple:.1f}" if 'both_accuracy_simple' in locals() else "N/A"
    ],
    'Tiempo/Pregunta (s)': [
        f"{total_evaluation_time / len(evaluation_results):.1f}" if 'total_evaluation_time' in locals() and evaluation_results else "N/A",
        f"{total_time_opt / len(results_optimized):.1f}" if 'total_time_opt' in locals() and results_optimized else "N/A",
        f"{total_time_simple / total_simple:.1f}" if 'total_time_simple' in locals() and total_simple > 0 else "N/A"
    ],
    'Muestra Evaluada': [
        f"{len(evaluation_results)}" if 'evaluation_results' in locals() else "N/A",
        f"{len(results_optimized)}" if 'results_optimized' in locals() else "N/A",
        f"{total_simple}" if 'total_simple' in locals() else "N/A"
    ]
}

df_final_comparison = pd.DataFrame(final_comparison)
print(df_final_comparison.to_string(index=False))


In [ ]:
# === VISUALIZACIÓN FINAL ===
print("\n🎨 Generando visualización final...")

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Análisis Final - Sistema RAG Legal (TFM)', fontsize=18, fontweight='bold', y=0.98)

# 1. Comparación de precisión "Ambos Correctos" entre configuraciones
if all(var in locals() for var in ['both_accuracy', 'both_accuracy_opt', 'both_accuracy_simple']):
    configs = ['Original\n(límite=20)', 'Optimizada\n(límite=15)', 'Simplificada\n(límite=15)']
    both_scores = [both_accuracy, both_accuracy_opt, both_accuracy_simple]
    colors_final = ['#3498DB', '#E74C3C', '#2ECC71']

    bars_final = ax1.bar(configs, both_scores, color=colors_final, alpha=0.8, edgecolor='black')
    ax1.set_title('Precisión "Ambos Correctos" por Configuración', fontweight='bold', fontsize=14)
    ax1.set_ylabel('Precisión (%)', fontsize=12)
    ax1.set_ylim(0, 100)
    ax1.grid(True, alpha=0.3)

    # Añadir valores y línea de objetivo
    for bar, score in zip(bars_final, both_scores):
        ax1.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 2,
                 f'{score:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

    ax1.axhline(y=70, color='red', linestyle='--', alpha=0.7, label='Objetivo 70%')
    ax1.legend()

# 2. Evolución de métricas
if all(var in locals() for var in ['law_accuracy', 'item_accuracy', 'both_accuracy']):
    metrics_evolution = ['Ley', 'Artículo', 'Ambos']
    original_vals = [law_accuracy, item_accuracy, both_accuracy]

    if 'law_accuracy_simple' in locals():
        simplified_vals = [law_accuracy_simple, item_accuracy_simple, both_accuracy_simple]

        x_metrics = np.arange(len(metrics_evolution))
        width_metrics = 0.35

        ax2.bar(x_metrics - width_metrics / 2, original_vals, width_metrics,
                label='Original', color='#3498DB', alpha=0.8)
        ax2.bar(x_metrics + width_metrics / 2, simplified_vals, width_metrics,
                label='Simplificada', color='#2ECC71', alpha=0.8)

        ax2.set_title('Comparación Detallada: Original vs Simplificada', fontweight='bold')
        ax2.set_ylabel('Precisión (%)')
        ax2.set_xticks(x_metrics)
        ax2.set_xticklabels(metrics_evolution)
        ax2.legend()
        ax2.grid(True, alpha=0.3)

# 3. Distribución de éxito por configuración (si tenemos datos)
if 'results_simplified' in locals() and results_simplified:
    success_distribution = {
        'Completamente exitoso': sum(1 for r in results_simplified if r['both_correct']),
        'Solo ley correcta': sum(1 for r in results_simplified if r['law_correct'] and not r['law_item_correct']),
        'Solo artículo correcto': sum(1 for r in results_simplified if not r['law_correct'] and r['law_item_correct']),
        'Sin éxito': sum(1 for r in results_simplified if not r['law_correct'] and not r['law_item_correct'])
    }

    labels_dist = list(success_distribution.keys())
    sizes_dist = list(success_distribution.values())
    colors_dist = ['#2ECC71', '#F39C12', '#E67E22', '#E74C3C']
    explode_dist = (0.1, 0, 0, 0)

    wedges, texts, autotexts = ax3.pie(sizes_dist, labels=labels_dist, colors=colors_dist,
                                       autopct='%1.1f%%', startangle=90, explode=explode_dist)
    ax3.set_title('Distribución de Resultados - Mejor Configuración', fontweight='bold')

    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')

# 4. Métricas de eficiencia
if all(var in locals() for var in ['total_evaluation_time', 'total_time_simple']):
    efficiency_data = {
        'Configuración': ['Original', 'Simplificada'],
        'Tiempo Total (min)': [total_evaluation_time / 60, total_time_simple / 60],
        'Preguntas': [len(evaluation_results) if evaluation_results else 0, total_simple],
        'Eficiencia (preg/min)': [
            len(evaluation_results) / (
                        total_evaluation_time / 60) if evaluation_results and total_evaluation_time > 0 else 0,
            total_simple / (total_time_simple / 60) if total_simple > 0 and total_time_simple > 0 else 0
        ]
    }

    ax4.bar(efficiency_data['Configuración'], efficiency_data['Eficiencia (preg/min)'],
            color=['#3498DB', '#2ECC71'], alpha=0.8)
    ax4.set_title('Eficiencia del Sistema (Preguntas/min)', fontweight='bold')
    ax4.set_ylabel('Preguntas por minuto')
    ax4.grid(True, alpha=0.3)

    for i, (config, eff) in enumerate(zip(efficiency_data['Configuración'], efficiency_data['Eficiencia (preg/min)'])):
        ax4.text(i, eff + 0.1, f'{eff:.1f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(f'data/final_analysis_{datetime.now().strftime("%Y%m%d_%H%M%S")}.png',
            dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# === CONCLUSIONES Y RECOMENDACIONES ===
print("\n🎯 CONCLUSIONES Y RECOMENDACIONES")
print("=" * 100)

# Identificar la mejor configuración
best_config = "N/A"
best_score = 0

if 'both_accuracy' in locals():
    best_config = "Original"
    best_score = both_accuracy

if 'both_accuracy_opt' in locals() and both_accuracy_opt > best_score:
    best_config = "Optimizada"
    best_score = both_accuracy_opt

if 'both_accuracy_simple' in locals() and both_accuracy_simple > best_score:
    best_config = "Simplificada"
    best_score = both_accuracy_simple

print(f"🏆 MEJOR CONFIGURACIÓN IDENTIFICADA:")
print(f"   • Configuración: {best_config}")
print(f"   • Precisión 'Ambos Correctos': {best_score:.1f}%")

print(f"\n📈 HALLAZGOS PRINCIPALES:")

if 'both_accuracy_simple' in locals() and 'both_accuracy' in locals():
    improvement = both_accuracy_simple - both_accuracy
    print(f"   • Las preguntas simplificadas mejoran la precisión en {improvement:+.1f} puntos porcentuales")

if 'both_accuracy_opt' in locals() and 'both_accuracy' in locals():
    opt_improvement = both_accuracy_opt - both_accuracy
    print(f"   • Reducir el límite de búsqueda cambia la precisión en {opt_improvement:+.1f} puntos porcentuales")

if 'results_simplified' in locals():
    avg_citations_simple = np.mean([r['num_citations'] for r in results_simplified])
    print(f"   • Promedio de citaciones con mejor configuración: {avg_citations_simple:.1f}")

print(f"\n🔧 RECOMENDACIONES PARA PRODUCCIÓN:")
print(f"   • Usar configuración '{best_config}' para máxima precisión")
print(f"   • Implementar preprocesamiento de queries para hacerlas más específicas")
print(f"   • Considerar ajuste dinámico del límite de búsqueda según complejidad de la query")

if best_score < 70:
    print(f"   ⚠️ La precisión actual ({best_score:.1f}%) está por debajo del objetivo del 70%")
    print(f"   • Recomendado: Revisar el modelo de embedding y la base de datos vectorial")
    print(f"   • Considerar fine-tuning del modelo de lenguaje con datos legales específicos")
else:
    print(f"   ✅ La precisión alcanzada ({best_score:.1f}%) supera el objetivo del 70%")

print(f"\n📚 LIMITACIONES DEL ESTUDIO:")
print(f"   • Evaluación limitada a {SAMPLE_SIZE if 'SAMPLE_SIZE' in locals() else 'N/A'} preguntas por configuración")
print(f"   • Dataset generado sintéticamente, podría no reflejar consultas reales")
print(f"   • Evaluación basada en coincidencia exacta de IDs, no en relevancia semántica")

print(f"\n🚀 TRABAJO FUTURO:")
print(f"   • Evaluar con dataset de consultas reales de usuarios")
print(f"   • Implementar métricas de relevancia semántica además de coincidencia exacta")
print(f"   • Explorar modelos de embedding especializados en dominio legal")
print(f"   • Desarrollar sistema de feedback de usuarios para mejora continua")

print(f"\n" + "=" * 100)
print(f"🎉 EVALUACIÓN DEL SISTEMA RAG LEGAL COMPLETADA")
print(f"📅 Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📊 Archivos generados disponibles en el directorio 'data/'")
print(f"=" * 100)